# Week 0 smoke (Kaggle T4×2)

1. **New notebook** → accelerator **GPU T4 × 2**. Never P100.
2. Turn **Internet** on.
3. Run all cells below. Do **not** download SmolLM2 or Qwen.

fp16 + GradScaler. Checkpoints go to `/kaggle/working/checkpoints/week0` so they survive a kernel interrupt. After the 10-minute cell, **interrupt that cell** (do not factory-reset), then run `--resume auto`. `curve.csv` must continue from the last step.

In [ ]:
import os
from pathlib import Path

REPO = "https://github.com/Caedral-ai/notrehybrid.git"
cwd = Path.cwd()

if (cwd / "setup_kaggle.sh").exists():
    root = cwd
elif (cwd / "notrehybrid" / "setup_kaggle.sh").exists():
    root = cwd / "notrehybrid"
else:
    !git clone --depth 1 {REPO} notrehybrid
    root = cwd / "notrehybrid"

os.chdir(root)
print("repo root:", root)
!bash setup_kaggle.sh

In [ ]:
from fla.layers import GatedDeltaNet
import torch

assert torch.cuda.get_device_capability()[0] >= 7
m = GatedDeltaNet(hidden_size=512, num_heads=8, mode="chunk").cuda().half()
x = torch.randn(2, 1024, 512, device="cuda", dtype=torch.float16)
assert m(x)[0].shape == x.shape
print("FLA OK", torch.cuda.get_device_name())

In [ ]:
!python -m pytest tests/test_hybrid_block.py -q

## Resume smoke

First a 3-step dry run (proves HybridBlock + save). Then ~10 minutes. **Interrupt** the long cell, then run `--resume auto` in this same session.

In [ ]:
CKPT = "/kaggle/working/checkpoints/week0"
!python -m notre.train.smoke_resume --minutes 1 --max-steps 3 --save-every 1 --ckpt-dir {CKPT}

In [ ]:
CKPT = "/kaggle/working/checkpoints/week0"
# Continues the dry run. Interrupt after ~10 min, then run the next cell.
!python -m notre.train.smoke_resume --minutes 10 --save-every 100 --keep-last 2 --ckpt-dir {CKPT} --resume auto

In [ ]:
CKPT = "/kaggle/working/checkpoints/week0"
!python -m notre.train.smoke_resume --minutes 2 --save-every 100 --keep-last 2 --ckpt-dir {CKPT} --resume auto
print("open", CKPT + "/curve.csv — steps must continue after the interrupt")